In [0]:
%sql
-- Paso 1 (estructura), mirada global: cantidad de columnas de cada tabla bronze
-- y sus primeras 6, para ubicar dónde está la columna de país, de año, etc.
SELECT table_name,
       COUNT(*) AS columnas,
       CONCAT_WS(', ', SLICE(ARRAY_SORT(COLLECT_LIST(STRUCT(ordinal_position, column_name))).column_name, 1, 6)) AS primeras_6
FROM happiness.information_schema.columns
WHERE table_schema = 'bronze'
GROUP BY table_name
ORDER BY table_name;

In [0]:
%sql
-- WHR · Paso 1 (estructura): cantidad de filas. Esperado: 2.116 (S3).
SELECT COUNT(*) AS filas FROM happiness.bronze.whr;

-- WHR · Paso 1 (estructura): tipos de columna. Todo STRING salvo _ingest_ts (D10/D11).
DESCRIBE TABLE happiness.bronze.whr;

-- WHR · Paso 1 (estructura): muestra para ver cómo viene el dato. WHR no trae código de país,
-- solo 'Country name', así que acá se define cómo se cruza con las demás fuentes.
SELECT * FROM happiness.bronze.whr LIMIT 5;

In [0]:
%sql
-- WHR · Paso 2 (nulos): ¿cómo se marcan los faltantes? Se miran las claves y las columnas numéricas
-- principales, contando NULL real, texto vacío y el texto 'null' por separado.
SELECT COUNT(*)                                         AS filas,
       COUNT_IF(`Year` IS NULL OR `Year` IN ('', 'null'))                          AS anio_faltante,
       COUNT_IF(`Country name` IS NULL OR `Country name` IN ('', 'null'))          AS pais_faltante,
       COUNT_IF(`Rank` IS NULL OR `Rank` IN ('', 'null'))                          AS rank_faltante,
       COUNT_IF(`Life evaluation (3-year average)` IS NULL OR `Life evaluation (3-year average)` IN ('', 'null')) AS puntaje_faltante,
       COUNT_IF(`Lower whisker` IS NULL OR `Lower whisker` IN ('', 'null'))        AS lower_faltante,
       COUNT_IF(`Upper whisker` IS NULL OR `Upper whisker` IN ('', 'null'))        AS upper_faltante,
       COUNT_IF(`Explained by: Log GDP per capita` IS NULL OR `Explained by: Log GDP per capita` IN ('', 'null')) AS pib_faltante,
       COUNT_IF(`Explained by: Generosity` IS NULL OR `Explained by: Generosity` IN ('', 'null'))                   AS generosidad_faltante,
       COUNT_IF(`Dystopia + residual` IS NULL OR `Dystopia + residual` IN ('', 'null'))                              AS distopia_faltante
FROM happiness.bronze.whr;

In [0]:
%sql
-- WHR · Pasos 3 y 6 (cardinalidad y temporal): filas y países por año.
-- Años como texto '2025.0': se convierten con TRY_CAST solo para ordenar y mostrar.
-- Referencia: el panel objetivo es de 2.089 países-año con puntaje WHR/OWID (00_contexto).
SELECT CAST(TRY_CAST(`Year` AS DOUBLE) AS INT) AS anio,
       COUNT(*)                                AS filas,
       COUNT(DISTINCT `Country name`)          AS paises
FROM happiness.bronze.whr
GROUP BY ALL
ORDER BY anio;

In [0]:
%sql
-- WHR · Paso 4 (rangos): el puntaje es una escala de 0 a 10. También se verifican dos reglas internas:
--  (a) lower_whisker <= puntaje <= upper_whisker.
--  (b) puntaje ≈ suma de las 6 contribuciones + 'Dystopia + residual' (tolerancia 0,01).
-- Si (a) o (b) fallan en alguna fila, hay un dato mal leído o un orden de columnas desplazado.
WITH n AS (
  SELECT CAST(TRY_CAST(`Year` AS DOUBLE) AS INT) AS anio,
         `Country name` AS pais,
         TRY_CAST(`Life evaluation (3-year average)` AS DOUBLE) AS puntaje,
         TRY_CAST(`Lower whisker` AS DOUBLE) AS lw,
         TRY_CAST(`Upper whisker` AS DOUBLE) AS uw,
         TRY_CAST(`Explained by: Log GDP per capita` AS DOUBLE)
         + TRY_CAST(`Explained by: Social support` AS DOUBLE)
         + TRY_CAST(`Explained by: Healthy life expectancy` AS DOUBLE)
         + TRY_CAST(`Explained by: Freedom to make life choices` AS DOUBLE)
         + TRY_CAST(`Explained by: Generosity` AS DOUBLE)
         + TRY_CAST(`Explained by: Perceptions of corruption` AS DOUBLE)
         + TRY_CAST(`Dystopia + residual` AS DOUBLE) AS suma
  FROM happiness.bronze.whr
)
SELECT MIN(puntaje) AS puntaje_min, MAX(puntaje) AS puntaje_max,
       COUNT_IF(puntaje IS NULL)                            AS puntaje_no_numerico,
       COUNT_IF(lw > puntaje OR puntaje > uw)               AS fuera_de_whiskers,
       COUNT_IF(ABS(suma - puntaje) > 0.01)                 AS suma_no_cierra
FROM n;

In [0]:
%sql
-- WHR · Paso 4 (rangos): rango por columna 'Explained by' (todas deberían ser >= 0).
SELECT MIN(TRY_CAST(`Explained by: Log GDP per capita` AS DOUBLE))             AS pib_min,
       MAX(TRY_CAST(`Explained by: Log GDP per capita` AS DOUBLE))             AS pib_max,
       MIN(TRY_CAST(`Explained by: Social support` AS DOUBLE))                 AS apoyo_min,
       MAX(TRY_CAST(`Explained by: Social support` AS DOUBLE))                 AS apoyo_max,
       MIN(TRY_CAST(`Explained by: Healthy life expectancy` AS DOUBLE))        AS salud_min,
       MAX(TRY_CAST(`Explained by: Healthy life expectancy` AS DOUBLE))        AS salud_max,
       MIN(TRY_CAST(`Explained by: Freedom to make life choices` AS DOUBLE))   AS libertad_min,
       MAX(TRY_CAST(`Explained by: Freedom to make life choices` AS DOUBLE))   AS libertad_max,
       MIN(TRY_CAST(`Explained by: Generosity` AS DOUBLE))                     AS generosidad_min,
       MAX(TRY_CAST(`Explained by: Generosity` AS DOUBLE))                     AS generosidad_max,
       MIN(TRY_CAST(`Explained by: Perceptions of corruption` AS DOUBLE))      AS corrupcion_min,
       MAX(TRY_CAST(`Explained by: Perceptions of corruption` AS DOUBLE))      AS corrupcion_max
FROM happiness.bronze.whr;

In [0]:
%sql
-- WHR · Paso 5 (duplicados): la clave natural es (Year, Country name).
-- ROW_NUMBER > 1 marca una clave repetida. Esperado: 0.
WITH numerado AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY `Year`, `Country name` ORDER BY _ingest_ts) AS rn
  FROM happiness.bronze.whr
)
SELECT COUNT(*) AS filas, COUNT_IF(rn > 1) AS claves_duplicadas FROM numerado;

In [0]:
%sql
-- WHR · Para dim_pais_base: nombres de país de WHR que no coinciden exactamente con ningún nombre
-- de Banco Mundial ni de WGI. Esa es la lista que habrá que resolver por tabla manual de equivalencias.
WITH whr AS (
  SELECT `Country name` AS nombre,
         COUNT(*) AS filas,
         MIN(CAST(TRY_CAST(`Year` AS DOUBLE) AS INT)) AS desde,
         MAX(CAST(TRY_CAST(`Year` AS DOUBLE) AS INT)) AS hasta
  FROM happiness.bronze.whr
  GROUP BY `Country name`
),
conocidos AS (
  SELECT `Country Name` AS nombre FROM happiness.bronze.banco_mundial
  UNION
  SELECT `Economy (name)` FROM happiness.bronze.wgi
)
SELECT w.nombre, w.filas, w.desde, w.hasta
FROM whr w LEFT ANTI JOIN conocidos c ON w.nombre = c.nombre
ORDER BY w.nombre;

### WHR pasa el EDA, con dos cosas para decidir.

**Lo bueno:** no hay duplicados en (año, país), ni nulos en año, país, rank o puntaje. El puntaje va de 1,36 a 7,86, dentro de la escala 0-10. Las contribuciones están entre 0 y su máximo, sin valores negativos.

Lo que hay que anotar:

- Falta 2013. Hay 14 años: 2011, 2012 y 2014 a 2025. El panel tiene 14 años, no 15.
- La descomposición está vacía en ~52% de las filas. Los whiskers faltan en 1.094 filas, las contribuciones "Explained by" en ~1.097 y Dystopia + residual en 1.103. Probablemente son los años viejos (WHR calculó la descomposición recién después).
- Los controles dieron 0, pero con un límite. fuera_de_whiskers y suma_no_cierra dieron 0 solo sobre las ~1.013 filas completas. Donde faltan datos la comparación no se evalúa, así que no es una validación de las 2.116.
- Nombres: 22 no coinciden con Banco Mundial ni WGI. Mi propuesta de equivalencias (código a confirmar) está en hallazgos_paises.md. Resumen:
    - 20 se resuelven con equivalencia manual. Por ejemplo, Congo → COG, DR Congo → COD, Türkiye → TUR, Swaziland → SWZ (hoy Eswatini), State of Palestine → PSE, Taiwan Province of China → TWN y Hong Kong SAR of China → HKG.
    - 2 quedan sin ISO3: North Cyprus (9 filas) y Somaliland Region (4 filas). Son los casos de la lista justificada del "hecho cuando".

In [0]:
%sql
-- WHR · Paso 2/6 (nulos por año): ¿en qué años faltan la descomposición y los whiskers?
-- Sirve para decidir si la descomposición (las 'Explained by') se usa en el análisis o solo el puntaje.
SELECT CAST(TRY_CAST(`Year` AS DOUBLE) AS INT) AS anio,
       COUNT(*)                                                  AS filas,
       COUNT_IF(`Lower whisker` IS NULL)                         AS sin_whiskers,
       COUNT_IF(`Explained by: Log GDP per capita` IS NULL)      AS sin_descomposicion,
       COUNT_IF(`Dystopia + residual` IS NULL)                   AS sin_distopia
FROM happiness.bronze.whr
GROUP BY ALL
ORDER BY anio;

In [0]:
%sql
-- WHR · Verificación de las equivalencias propuestas: ¿existen estos códigos en Banco Mundial?
-- Se muestra el nombre que usa Banco Mundial para ver que corresponde al país de WHR.
-- Las filas que falten acá son las que solo están en WGI (por ejemplo TWN).
SELECT DISTINCT `Country Code` AS codigo, `Country Name` AS nombre_bm
FROM happiness.bronze.banco_mundial
WHERE `Country Code` IN ('COG','COD','CIV','EGY','GMB','HKG','IRN','KGZ','PRI','KOR',
                         'MDA','SVK','SOM','PSE','SWZ','SYR','TWN','TUR','VEN','YEM')
ORDER BY codigo;

## Conclusiones WHR (bronze.whr, 2.116 filas)
- Clave (Year, Country name) sin duplicados. Sin nulos en año, país, rank ni puntaje.
- Year y Rank vienen como texto con '.0' (ej. '2025.0'): Silver convierte con CAST a DOUBLE y luego INT.
- 14 años: 2011, 2012 y 2014-2025 (falta 2013). 137-158 países por año.
- Puntaje 1,364-7,856 (escala 0-10). Contribuciones >= 0.
- Whiskers y descomposición (Explained by, Dystopia) vacíos en ~52% de las filas (ver detalle por año).
- Controles internos (puntaje dentro de whiskers; suma de contribuciones = puntaje) sin fallas en las filas completas.
- Para dim_pais_base: sin código de país, cruce por nombre; 22 nombres sin coincidencia exacta (20 equivalencias manuales, 2 sin ISO3: North Cyprus y Somaliland Region). Detalle en eda/hallazgos_paises.md.
- Descomposición (whiskers, Explained by, Dystopia): 100% vacía en 2011-2018, casi completa desde 2019. El puntaje cubre todo el panel; la descomposición oficial, solo 2019-2025.
- Equivalencias de nombre → ISO3 verificadas contra Banco Mundial/WGI: 20 confirmadas, 2 sin ISO3 (North Cyprus, Somaliland Region).